In [0]:
RAW_PATH = "/Volumes/cx/default/cx_project_raw"

In [0]:
CATALOG = "cx"          
SCHEMA_BRONZE = "cx_bronze"
SCHEMA_SILVER = "cx_silver"

In [0]:
# Create catalog/schemas if they don't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS cx.{SCHEMA_BRONZE}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS cx.{SCHEMA_SILVER}")
print(f"Schemas ready: cx.{SCHEMA_BRONZE}, cx.{SCHEMA_SILVER}")

Schemas ready: cx.cx_bronze, cx.cx_silver


In [0]:
from pyspark.sql.functions import current_timestamp, input_file_name, lit

def ingest_to_bronze(file_name: str, table_name: str):
    """Read raw CSV and write to Bronze Delta table."""
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "false")  # everything as string in Bronze
        .csv(f"{RAW_PATH}/{file_name}")
        .withColumn("ingestion_timestamp", current_timestamp())
        .withColumn("source_file", lit(file_name))
    )
    full_table = f"{CATALOG}.{SCHEMA_BRONZE}.{table_name}"
    df.write.mode("overwrite").format("delta").saveAsTable(full_table)
    count = spark.table(full_table).count()
    print(f"  {full_table}: {count:,} rows")
    return count


In [0]:
print("Ingesting Bronze tables...")
ingest_to_bronze("customers.csv", "customers")
ingest_to_bronze("service_interactions.csv", "service_interactions")
ingest_to_bronze("product_usage.csv", "product_usage")
ingest_to_bronze("surveys.csv", "surveys")

Ingesting Bronze tables...
  cx.cx_bronze.customers: 50,050 rows
  cx.cx_bronze.service_interactions: 350,550 rows
  cx.cx_bronze.product_usage: 1,160,000 rows
  cx.cx_bronze.surveys: 29,745 rows


29745

In [0]:
from pyspark.sql.functions import (
    col, to_date, try_to_date, to_timestamp, when, trim, initcap, lower,
    regexp_replace, coalesce, lit
)
from pyspark.sql.types import (
    StringType, IntegerType, DoubleType, DateType, TimestampType
)

dq_log = []  # collect DQ findings to surface at the end

def log_dq(table: str, check: str, count: int, action: str):
    dq_log.append({"table": table, "check": check, "rows_affected": count, "action": action})

In [0]:
bronze_customers = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.customers")

In [0]:
total_in = bronze_customers.count()
deduped = bronze_customers.dropDuplicates(["customer_id"])
total_out = deduped.count()
log_dq("customers", "duplicate customer_id removed", total_in - total_out, "dropped duplicates")

In [0]:
silver_customers = (
    deduped
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("customer_name", trim(col("customer_name")))
    .withColumn("customer_type", trim(col("customer_type")))
    .withColumn("region", trim(col("region")))
    .withColumn("country", trim(col("country")))
    # Normalize service_tier casing (Gold/Silver/Bronze)
    .withColumn("service_tier", initcap(trim(col("service_tier"))))
    .withColumn("contract_start_date", to_date(col("contract_start_date")))
    .withColumn("annual_recurring_revenue_usd", col("annual_recurring_revenue_usd").cast(DoubleType()))
    .withColumn("employee_count", col("employee_count").cast(IntegerType()))
    .withColumn("primary_contact_email", trim(col("primary_contact_email")))
    # Quality flags
    .withColumn("dq_flag_missing_email", when(col("primary_contact_email").isNull(), 1).otherwise(0))
    .withColumn("dq_flag_missing_type", when(col("customer_type").isNull(), 1).otherwise(0))
)


In [0]:
missing_email = silver_customers.filter(col("dq_flag_missing_email") == 1).count()
missing_type = silver_customers.filter(col("dq_flag_missing_type") == 1).count()
log_dq("customers", "missing primary_contact_email", missing_email, "flagged, kept")
log_dq("customers", "missing customer_type", missing_type, "flagged, kept")

silver_customers.write.mode("overwrite").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_SILVER}.customers"
)
print(f"  Silver customers: {silver_customers.count():,} rows")

  Silver customers: 50,000 rows


In [0]:
bronze_tickets = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.service_interactions")
valid_customer_ids = spark.table(f"{CATALOG}.{SCHEMA_SILVER}.customers").select("customer_id")

silver_tickets = (
    bronze_tickets
    .withColumn("ticket_id", trim(col("ticket_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("ticket_open_date", to_timestamp(col("ticket_open_date")))
    .withColumn("channel", trim(col("channel")))
    .withColumn("issue_category", trim(col("issue_category")))
    .withColumn("priority", trim(col("priority")))
    .withColumn("first_response_minutes", col("first_response_minutes").cast(DoubleType()))
    .withColumn("resolution_hours", col("resolution_hours").cast(DoubleType()))
    .withColumn("escalated_flag", col("escalated_flag").cast(DoubleType()).cast(IntegerType()))
    .withColumn("csat_post_ticket", col("csat_post_ticket").cast(DoubleType()).cast(IntegerType()))
)

In [0]:
neg_res = silver_tickets.filter(col("resolution_hours") < 0).count()
silver_tickets = silver_tickets.withColumn(
    "resolution_hours",
    when(col("resolution_hours") < 0, None).otherwise(col("resolution_hours"))
)
log_dq("service_interactions", "negative resolution_hours", neg_res, "set to NULL")



In [0]:
# DQ: orphan customer_ids (referential integrity)
orphans = silver_tickets.join(valid_customer_ids, "customer_id", "left_anti").count()
log_dq("service_interactions", "orphan customer_id (not in customers)", orphans, "flagged via dq_flag_orphan")

silver_tickets = silver_tickets.withColumn(
    "dq_flag_orphan",
    when(
        col("customer_id").isin([row.customer_id for row in valid_customer_ids.collect()]),
        0
    ).otherwise(1)
)

In [0]:
silver_tickets.write.mode("overwrite").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_SILVER}.service_interactions"
)
print(f"  Silver service_interactions: {silver_tickets.count():,} rows")

  Silver service_interactions: 350,550 rows


In [0]:
bronze_usage = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.product_usage")

silver_usage = (
    bronze_usage
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn("snapshot_month", to_date(col("snapshot_month")))
    .withColumn("active_instruments", col("active_instruments").cast(IntegerType()))
    .withColumn("test_volume", col("test_volume").cast(IntegerType()))
    .withColumn("uptime_pct", col("uptime_pct").cast(DoubleType()))
    .withColumn("error_events_count", col("error_events_count").cast(IntegerType()))
    .withColumn("software_version", trim(col("software_version")))
    .withColumn("avg_test_runtime_minutes", col("avg_test_runtime_minutes").cast(DoubleType()))
)


In [0]:
over_one = silver_usage.filter(col("uptime_pct") > 1.0).count()
silver_usage = silver_usage.withColumn(
    "uptime_pct",
    when(col("uptime_pct") > 1.0, 1.0).otherwise(col("uptime_pct"))
)
log_dq("product_usage", "uptime_pct > 1.0", over_one, "capped at 1.0")

In [0]:
total_in = silver_usage.count()
silver_usage = silver_usage.dropDuplicates(["customer_id", "snapshot_month"])
total_out = silver_usage.count()
log_dq("product_usage", "duplicate (customer_id, snapshot_month)", total_in - total_out, "dropped duplicates")

silver_usage.write.mode("overwrite").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_SILVER}.product_usage"
)
print(f"  Silver product_usage: {silver_usage.count():,} rows")

  Silver product_usage: 1,160,000 rows


In [0]:
bronze_surveys = spark.table(f"{CATALOG}.{SCHEMA_BRONZE}.surveys")


In [0]:
silver_surveys = (
    bronze_surveys
    .withColumn("survey_id", trim(col("survey_id")))
    .withColumn("customer_id", trim(col("customer_id")))
    .withColumn(
        "survey_date",
        coalesce(
            try_to_date(col("survey_date"), "yyyy-MM-dd"),
            try_to_date(col("survey_date"), "MM/dd/yyyy"),
        )
    )
    .withColumn("nps_score", col("nps_score").cast(IntegerType()))
    .withColumn("csat_score", col("csat_score").cast(IntegerType()))
    .withColumn("ces_score", col("ces_score").cast(IntegerType()))
    .withColumn("open_comment", trim(col("open_comment")))
)

In [0]:
oor = silver_surveys.filter((col("nps_score") < 0) | (col("nps_score") > 10)).count()
silver_surveys = silver_surveys.withColumn(
    "nps_score",
    when((col("nps_score") < 0) | (col("nps_score") > 10), None).otherwise(col("nps_score"))
)
log_dq("surveys", "NPS score out of range (0-10)", oor, "set to NULL")


In [0]:
silver_surveys = silver_surveys.withColumn(
    "nps_bucket",
    when(col("nps_score") <= 6, "Detractor")
    .when(col("nps_score") <= 8, "Passive")
    .when(col("nps_score") <= 10, "Promoter")
    .otherwise(None)
)

silver_surveys.write.mode("overwrite").format("delta").saveAsTable(
    f"{CATALOG}.{SCHEMA_SILVER}.surveys"
)
print(f"  Silver surveys: {silver_surveys.count():,} rows")


  Silver surveys: 29,745 rows


In [0]:
import pandas as pd

if dq_log:
    dq_df = pd.DataFrame(dq_log)
    print("=== Data Quality Findings ===")
    print(dq_df.to_string(index=False))
    
    # Optionally persist the DQ log as a Delta table for ongoing monitoring
    spark.createDataFrame(dq_df).write.mode("overwrite").format("delta").saveAsTable(
        f"{CATALOG}.{SCHEMA_SILVER}.dq_log_week1"
    )
else:
    print("=== Data Quality Findings ===")
    print("No data quality issues logged. Either all checks passed or cells were executed out of order.")
    print("To populate the DQ log, re-run cells 7-21 in order.")

=== Data Quality Findings ===
               table                                   check  rows_affected                     action
           customers           duplicate customer_id removed             50         dropped duplicates
           customers           missing primary_contact_email           1002              flagged, kept
           customers                   missing customer_type            471              flagged, kept
service_interactions               negative resolution_hours           3329                set to NULL
service_interactions   orphan customer_id (not in customers)            746 flagged via dq_flag_orphan
       product_usage                        uptime_pct > 1.0           3419              capped at 1.0
       product_usage duplicate (customer_id, snapshot_month)              0         dropped duplicates
             surveys           NPS score out of range (0-10)            156                set to NULL


In [0]:
## 5. Verification queries

In [0]:
%sql
SELECT 'customers' AS table, COUNT(*) AS n FROM cx.cx_silver.customers
UNION ALL
SELECT 'service_interactions', COUNT(*) FROM cx.cx_silver.service_interactions
UNION ALL
SELECT 'product_usage', COUNT(*) FROM cx.cx_silver.product_usage
UNION ALL
SELECT 'surveys', COUNT(*) FROM cx.cx_silver.surveys

table,n
customers,50000
service_interactions,350550
product_usage,1160000
surveys,29745


In [0]:
%sql
SELECT service_tier, COUNT(*) AS customers
FROM cx.cx_silver.customers
GROUP BY service_tier

service_tier,customers
Bronze,27373
Gold,7603
Silver,15024


In [0]:
 %sql
 -- NPS bucket distribution
SELECT nps_bucket, COUNT(*) AS responses
FROM cx.cx_silver.surveys
WHERE nps_bucket IS NOT NULL
GROUP BY nps_bucket
ORDER BY responses DESC

nps_bucket,responses
Detractor,10962
Promoter,9781
Passive,8846
